# Approach3 UTM Zone-Row Window Coverage Pre-analysis

This notebook tests temporal-window requirements by UTM zone-row cell. It is intended to answer whether the minimum `+/-N` day window changes systematically with latitude bands, while still preserving the actual local product coverage for each UTM cell.

## Context & Methods

The notebook uses the same coverage logic as the single-date and multidate pre-analysis notebooks:

- Dynamic World is the first source.
- OPERA DSWx-HLS Landsat fills only pixels where Dynamic World is invalid.
- OPERA DSWx-S1 fills only remaining pixels where both DW and HLS are invalid.
- The scan tests `+/-0` through `+/-MAX_WINDOW_DAYS` around each reference date.

Two spatial modes are supported:

- `single_zone`: test all selected latitude rows inside one UTM zone, for example UTM zone 34 rows C-X.
- `continent`: test all selected UTM cells intersecting one continent, for example Africa, optionally clipped to the continent geometry.

Important caveat: this is a diagnostic. Latitude is only one driver of acquisition frequency. Clouds, product availability, orbital geometry, land/water masks, and local data gaps can still make the optimal window differ between places at the same latitude.

## 1. Parameters

Defaults are conservative: one UTM zone, one reference date, and windows up to `+/-10` days. For the full Africa/year test, switch `ANALYSIS_MODE` to `continent` and `TARGET_DATE_MODE` to `cadence`.

In [1]:
# Earth Engine
EE_PROJECT = "hardy-tenure-383607"
USE_HIGH_VOLUME_ENDPOINT = False

# UTM grid shapefile zip.
UTM_GRID_ZIP = r"C:\Users\ibana\Downloads\World_UTM_Grid_-4083655944859593094.zip"

# Spatial mode: "single_zone" or "continent".
ANALYSIS_MODE = "single_zone"
UTM_ZONE = 34
UTM_ROWS = None  # None means all standard UTM latitude rows C-X. Example: ["Q", "R", "S", "T"]
INCLUDE_POLAR_ROWS = False  # Usually keep False: rows A, B, Y, Z and zone 0 are not standard UTM zones.

# Continent mode settings.
CONTINENT_NAME = "Africa"
CONTINENT_COLLECTION = "USDOS/LSIB_SIMPLE/2017"
CONTINENT_PROPERTY = "wld_rgn"
CONTINENT_VALUE = "Africa"
CLIP_CELLS_TO_CONTINENT = True
CONTINENT_PREFILTER_BBOX = (-26.0, -36.0, 60.0, 39.0)  # lon_min, lat_min, lon_max, lat_max for Africa.
MIN_CELL_AREA_KM2 = 1000.0

# Reference dates.
START_DATE = "2025-01-01"
END_DATE = "2026-01-01"  # Exclusive.
TARGET_DATE_MODE = "manual"  # "manual" or "cadence".
MANUAL_TARGET_DATES = ["2025-05-01"]
PRIMARY_WINDOW_DAYS = 5
MAX_REFERENCE_DATES = None  # Set an integer for a shorter continent smoke test.

# Window scan settings.
MAX_WINDOW_DAYS = 30
WINDOW_STEP_DAYS = 1
COVERAGE_TARGET_PCT = 99.0
AREA_SCALE_M = 300
REDUCE_TILE_SCALE = 4
INCLUDE_IMAGE_COUNTS = True
REUSE_EXISTING_WINDOW_SCAN = True

# Large-run controls.
MAX_GRID_CELLS = None  # Set an integer for a smoke test.
SKIP_CELL_AREA_CHECK = False  # Keep False when clipping to continent.

# Source settings.
INCLUDE_HLS_SENTINEL2 = False  # Keep False: Dynamic World already uses Sentinel-2.

# Output settings. None builds a timestamped label.
OUTPUT_RUN_LABEL = None

## 2. Setup

This initializes Earth Engine and creates the run output folders.

In [2]:
from __future__ import annotations

from datetime import date, datetime, timedelta, timezone
from pathlib import Path
import json
import math
import os
import re
import sys
import zipfile

def configure_gdal_environment() -> None:
    """Point GDAL/PROJ-dependent packages to data bundled in the active env."""
    env_root_candidates = [Path(sys.prefix), Path(sys.executable).resolve().parent]
    for env_root in env_root_candidates:
        gdal_data = env_root / "Library" / "share" / "gdal"
        if gdal_data.exists():
            os.environ.setdefault("GDAL_DATA", str(gdal_data))
            break
    for env_root in env_root_candidates:
        proj_data = env_root / "Library" / "share" / "proj"
        if proj_data.exists():
            os.environ.setdefault("PROJ_LIB", str(proj_data))
            break


configure_gdal_environment()

import ee
import geemap
import geopandas as gpd
import pandas as pd
from IPython.display import display
from shapely.geometry import box, mapping

cwd = Path.cwd().resolve()
if cwd.name == "Approach3":
    APPROACH3_ROOT = cwd
elif (cwd / "Approaches" / "Approach3").exists():
    APPROACH3_ROOT = cwd / "Approaches" / "Approach3"
else:
    APPROACH3_ROOT = next(parent for parent in cwd.parents if parent.name == "Approach3")

sys.path.insert(0, str(APPROACH3_ROOT / "src"))

from sw_dws1_approach3.datasets import (
    DynamicWorldThresholds,
    OperaHlsWtrClass,
    OperaS1WtrClass,
    dynamic_world_collection,
    opera_dswx_hls_collection,
    opera_dswx_s1_collection,
)
from sw_dws1_approach3.gee_session import initialize_earth_engine
from sw_dws1_approach3.periods import validate_date_window

if EE_PROJECT == "your-google-cloud-project-id":
    raise ValueError("Set EE_PROJECT before running the notebook.")

validate_date_window(START_DATE, END_DATE)
if MAX_WINDOW_DAYS < 0:
    raise ValueError("MAX_WINDOW_DAYS must be non-negative.")
if WINDOW_STEP_DAYS <= 0:
    raise ValueError("WINDOW_STEP_DAYS must be positive.")

initialize_earth_engine(project=EE_PROJECT, use_high_volume_endpoint=USE_HIGH_VOLUME_ENDPOINT)

def safe_label(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_") or "run"

mode_label = f"utm_zone_{UTM_ZONE}" if ANALYSIS_MODE == "single_zone" else f"continent_{CONTINENT_NAME}"
run_label = OUTPUT_RUN_LABEL or f"{safe_label(mode_label)}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = APPROACH3_ROOT / "notebooks" / "outputs" / "utm_zone_row_window_preanalysis" / run_label
CSV_DIR = OUTPUT_DIR / "csv"
FIGURE_DIR = OUTPUT_DIR / "figures"
for folder in [OUTPUT_DIR, CSV_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Approach3 root:", APPROACH3_ROOT)
print("Output directory:", OUTPUT_DIR)

Approach3 root: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3
Output directory: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\utm_zone_row_window_preanalysis\utm_zone_34_20260703_120513


## 3. Load UTM Grid

The shapefile is read from the zip and reprojected to WGS84. Standard UTM rows are C-X, excluding I and O.

In [3]:
STANDARD_UTM_ROWS = list("CDEFGHJKLMNPQRSTUVWX")
ROW_ORDER = {row: i for i, row in enumerate(STANDARD_UTM_ROWS)}

zip_path = Path(UTM_GRID_ZIP)
if not zip_path.exists():
    raise FileNotFoundError(f"UTM grid zip not found: {zip_path}")

with zipfile.ZipFile(zip_path) as z:
    shapefile_names = [name for name in z.namelist() if name.lower().endswith(".shp")]
if len(shapefile_names) != 1:
    raise ValueError(f"Expected one .shp inside {zip_path}, found: {shapefile_names}")

grid = gpd.read_file("zip://" + str(zip_path).replace("\\", "/"), engine="pyogrio")
required_columns = {"ZONE", "ROW_", "geometry"}
missing_columns = required_columns - set(grid.columns)
if missing_columns:
    raise ValueError(f"UTM grid is missing required columns: {sorted(missing_columns)}")

grid = grid.to_crs("EPSG:4326")
grid["utm_zone"] = grid["ZONE"].astype(float).round().astype(int)
grid["utm_row"] = grid["ROW_"].astype(str).str.strip()
grid["cell_id"] = grid.apply(lambda row: f"Z{int(row['utm_zone']):02d}{row['utm_row']}", axis=1)

bounds = grid.geometry.bounds
grid["lon_min"] = bounds["minx"]
grid["lat_min"] = bounds["miny"]
grid["lon_max"] = bounds["maxx"]
grid["lat_max"] = bounds["maxy"]
grid["lat_center"] = (grid["lat_min"] + grid["lat_max"]) / 2
grid["row_order"] = grid["utm_row"].map(ROW_ORDER)

print("Rows in UTM grid:", len(grid))
print("CRS:", grid.crs)
display(grid.drop(columns="geometry").head(20))

Rows in UTM grid: 1201
CRS: EPSG:4326


,ZONE,ROW_,WEST_VALUE,CM_VALUE,EAST_VALUE,utm_zone,utm_row,cell_id,lon_min,lat_min,lon_max,lat_max,lat_center,row_order
0,0.0,Y,NoZN,NoZN,NoZN,0,Y,Z00Y,-1.800000e+02,84.0,8.381906e-08,89.0,86.5,NaN
1,22.0,P,54W,51W,48W,22,P,Z22P,-5.400000e+01,8.0,-4.800000e+01,16.0,12.0,11.0
2,0.0,Z,NoZN,NoZN,NoZN,0,Z,Z00Z,8.381903e-08,84.0,1.800000e+02,89.0,86.5,NaN
3,1.0,X,180W,177W,174W,1,X,Z01X,-1.800000e+02,72.0,-1.740000e+02,84.0,78.0,19.0
4,2.0,X,174W,171W,168W,2,X,Z02X,-1.740000e+02,72.0,-1.680000e+02,84.0,78.0,19.0
5,23.0,P,48W,45W,42W,23,P,Z23P,-4.800000e+01,8.0,-4.200000e+01,16.0,12.0,11.0
6,3.0,X,168W,165W,162W,3,X,Z03X,-1.680000e+02,72.0,-1.620000e+02,84.0,78.0,19.0
7,24.0,P,42W,39W,36W,24,P,Z24P,-4.200000e+01,8.0,-3.600000e+01,16.0,12.0,11.0
8,4.0,X,162W,159W,156W,4,X,Z04X,-1.620000e+02,72.0,-1.560000e+02,84.0,78.0,19.0
9,25.0,P,36W,33W,30W,25,P,Z25P,-3.600000e+01,8.0,-3.000000e+01,16.0,12.0,11.0


## 4. Select UTM Cells

This builds the spatial units to scan. In continent mode the local bbox is only a prefilter; final clipping is done in Earth Engine.

In [4]:
def selected_rows() -> list[str]:
    if UTM_ROWS is None:
        return STANDARD_UTM_ROWS if not INCLUDE_POLAR_ROWS else sorted(grid["utm_row"].dropna().unique().tolist())
    return [str(row).strip().upper() for row in UTM_ROWS]


rows_to_use = selected_rows()
candidate_grid = grid.copy()
if not INCLUDE_POLAR_ROWS:
    candidate_grid = candidate_grid[
        (candidate_grid["utm_zone"].between(1, 60))
        & (candidate_grid["utm_row"].isin(STANDARD_UTM_ROWS))
    ].copy()
candidate_grid = candidate_grid[candidate_grid["utm_row"].isin(rows_to_use)].copy()

if ANALYSIS_MODE == "single_zone":
    selected_grid = candidate_grid[candidate_grid["utm_zone"] == int(UTM_ZONE)].copy()
elif ANALYSIS_MODE == "continent":
    if CONTINENT_PREFILTER_BBOX is None:
        selected_grid = candidate_grid.copy()
    else:
        bbox_geom = box(*CONTINENT_PREFILTER_BBOX)
        selected_grid = candidate_grid[candidate_grid.intersects(bbox_geom)].copy()
else:
    raise ValueError('ANALYSIS_MODE must be "single_zone" or "continent".')

selected_grid = selected_grid.sort_values(["utm_zone", "row_order", "utm_row"]).reset_index(drop=True)
if MAX_GRID_CELLS is not None:
    selected_grid = selected_grid.head(int(MAX_GRID_CELLS)).copy()
if selected_grid.empty:
    raise ValueError("No UTM cells selected. Check ANALYSIS_MODE, UTM_ZONE, UTM_ROWS, or continent bbox.")

selected_cells_geojson = OUTPUT_DIR / "selected_utm_cells.geojson"
selected_grid.to_file(selected_cells_geojson, driver="GeoJSON")

print("Selected UTM cells:", len(selected_grid))
print("Saved:", selected_cells_geojson)
display(selected_grid[["cell_id", "utm_zone", "utm_row", "lat_min", "lat_max", "lon_min", "lon_max"]].head(40))

Selected UTM cells: 19
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\utm_zone_row_window_preanalysis\utm_zone_34_20260703_120513\selected_utm_cells.geojson


,cell_id,utm_zone,utm_row,lat_min,lat_max,lon_min,lon_max
0,Z34C,34,C,-8.000000e+01,-7.200000e+01,18.0,24.0
1,Z34D,34,D,-7.200000e+01,-6.400000e+01,18.0,24.0
2,Z34E,34,E,-6.400000e+01,-5.600000e+01,18.0,24.0
3,Z34F,34,F,-5.600000e+01,-4.800000e+01,18.0,24.0
4,Z34G,34,G,-4.800000e+01,-4.000000e+01,18.0,24.0
5,Z34H,34,H,-4.000000e+01,-3.200000e+01,18.0,24.0
6,Z34J,34,J,-3.200000e+01,-2.400000e+01,18.0,24.0
7,Z34K,34,K,-2.400000e+01,-1.600000e+01,18.0,24.0
8,Z34L,34,L,-1.600000e+01,-8.000000e+00,18.0,24.0
9,Z34M,34,M,-8.000000e+00,1.257285e-07,18.0,24.0


## 5. Reference Dates

Use `manual` for one or a few dates, or `cadence` for a full-year non-overlapping sequence using the primary window stride.

In [5]:
def _parse_date(value: str) -> date:
    return date.fromisoformat(value)


def cadence_dates(start_date: str, end_date: str, primary_window_days: int, max_dates: int | None) -> list[str]:
    stride_days = 2 * primary_window_days + 1
    current = _parse_date(start_date)
    end = _parse_date(end_date)
    out = []
    while current < end:
        out.append(current.isoformat())
        if max_dates is not None and len(out) >= int(max_dates):
            break
        current += timedelta(days=stride_days)
    return out


if TARGET_DATE_MODE == "manual":
    target_dates = [str(value) for value in MANUAL_TARGET_DATES]
    if MAX_REFERENCE_DATES is not None:
        target_dates = target_dates[: int(MAX_REFERENCE_DATES)]
elif TARGET_DATE_MODE == "cadence":
    target_dates = cadence_dates(START_DATE, END_DATE, PRIMARY_WINDOW_DAYS, MAX_REFERENCE_DATES)
else:
    raise ValueError('TARGET_DATE_MODE must be "manual" or "cadence".')

if not target_dates:
    raise ValueError("No target dates selected.")

target_df = pd.DataFrame({"target_date": target_dates})
target_csv = CSV_DIR / "target_dates.csv"
target_df.to_csv(target_csv, index=False)

print("Target dates:", len(target_dates))
print("Saved:", target_csv)
display(target_df.head(40))

Target dates: 1
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\utm_zone_row_window_preanalysis\utm_zone_34_20260703_120513\csv\target_dates.csv


,target_date
0,2025-05-01


## 6. Earth Engine Geometry Helpers

This clips selected cells to the continent when requested and stores the effective AOI area used for percentages.

In [6]:
def continent_geometry() -> ee.Geometry | None:
    if ANALYSIS_MODE != "continent" or not CLIP_CELLS_TO_CONTINENT:
        return None
    continent_fc = ee.FeatureCollection(CONTINENT_COLLECTION).filter(
        ee.Filter.eq(CONTINENT_PROPERTY, CONTINENT_VALUE)
    )
    continent_feature_count = int(continent_fc.size().getInfo())
    if continent_feature_count == 0:
        raise ValueError(
            "The continent filter returned zero features. Check CONTINENT_COLLECTION, "
            "CONTINENT_PROPERTY, and CONTINENT_VALUE."
        )
    print(f"Continent clipping features: {continent_feature_count}")
    return continent_fc.geometry()


CONTINENT_GEOMETRY = continent_geometry()


def ee_geometry_from_shapely(geometry) -> ee.Geometry:
    return ee.Geometry(mapping(geometry), proj="EPSG:4326", geodesic=False)


def effective_cell_geometry(geometry) -> ee.Geometry:
    ee_geom = ee_geometry_from_shapely(geometry)
    if CONTINENT_GEOMETRY is not None:
        ee_geom = ee_geom.intersection(CONTINENT_GEOMETRY, ee.ErrorMargin(1000))
    return ee_geom


def geometry_area_km2(geometry: ee.Geometry) -> float:
    return float(geometry.area(maxError=1000).divide(1_000_000).getInfo())


cell_records = []
for _, row in selected_grid.iterrows():
    ee_geom = effective_cell_geometry(row.geometry)
    area_km2 = None if SKIP_CELL_AREA_CHECK else geometry_area_km2(ee_geom)
    if area_km2 is not None and area_km2 < MIN_CELL_AREA_KM2:
        continue
    cell_records.append({
        "cell_id": row["cell_id"],
        "utm_zone": int(row["utm_zone"]),
        "utm_row": row["utm_row"],
        "row_order": row["row_order"],
        "lat_min": row["lat_min"],
        "lat_max": row["lat_max"],
        "lat_center": row["lat_center"],
        "lon_min": row["lon_min"],
        "lon_max": row["lon_max"],
        "aoi_area_km2": area_km2,
        "geometry": row.geometry,
    })

if not cell_records:
    raise ValueError("No selected UTM cells have enough effective area after optional continent clipping.")

cell_df = pd.DataFrame([{k: v for k, v in record.items() if k != "geometry"} for record in cell_records])
cell_metadata_csv = CSV_DIR / "selected_cell_metadata.csv"
cell_df.to_csv(cell_metadata_csv, index=False)
print("Cells after optional clipping/area filter:", len(cell_df))
print("Saved:", cell_metadata_csv)
display(cell_df.head(40))

Cells after optional clipping/area filter: 19
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\utm_zone_row_window_preanalysis\utm_zone_34_20260703_120513\csv\selected_cell_metadata.csv


,cell_id,utm_zone,utm_row,row_order,lat_min,lat_max,lat_center,lon_min,lon_max,aoi_area_km2
0,Z34C,34,C,0.0,-8.000000e+01,-7.200000e+01,-76.0,18.0,24.0,143454.618376
1,Z34D,34,D,1.0,-7.200000e+01,-6.400000e+01,-68.0,18.0,24.0,222135.241375
2,Z34E,34,E,2.0,-6.400000e+01,-5.600000e+01,-60.0,18.0,24.0,296493.826817
3,Z34F,34,F,3.0,-5.600000e+01,-4.800000e+01,-52.0,18.0,24.0,365083.209922
4,Z34G,34,G,4.0,-4.800000e+01,-4.000000e+01,-44.0,18.0,24.0,426568.251207
5,Z34H,34,H,5.0,-4.000000e+01,-3.200000e+01,-36.0,18.0,24.0,479751.771408
6,Z34J,34,J,6.0,-3.200000e+01,-2.400000e+01,-28.0,18.0,24.0,523598.000938
7,Z34K,34,K,7.0,-2.400000e+01,-1.600000e+01,-20.0,18.0,24.0,557252.799035
8,Z34L,34,L,8.0,-1.600000e+01,-8.000000e+00,-12.0,18.0,24.0,580060.467861
9,Z34M,34,M,9.0,-8.000000e+00,1.257285e-07,-4.0,18.0,24.0,591576.515462


## 7. Coverage Helpers

These functions compute valid coverage and source contribution for one UTM cell, target date, and window size.

In [7]:
thresholds = DynamicWorldThresholds()


def dw_valid_observation(image: ee.Image) -> ee.Image:
    image = ee.Image(image)
    water = image.select("water")
    flooded = image.select("flooded_vegetation")
    valid = water.gt(thresholds.water).Or(water.lte(thresholds.nonwater)).Or(
        flooded.gt(thresholds.flooded_vegetation)
    )
    return valid.rename("valid").toByte().updateMask(water.mask())


def hls_valid_observation(image: ee.Image) -> ee.Image:
    wtr = ee.Image(image).select("WTR_Water_classification")
    return wtr.lt(OperaHlsWtrClass.SNOW_ICE).rename("valid").toByte().updateMask(wtr.mask())


def s1_valid_observation(image: ee.Image) -> ee.Image:
    wtr = ee.Image(image).select("WTR_Water_classification")
    return wtr.lt(OperaS1WtrClass.HAND_MASKED).rename("valid").toByte().updateMask(wtr.mask())


def window_bounds(target_date: str, window_days: int) -> tuple[str, str, str]:
    target = _parse_date(target_date)
    start = target - timedelta(days=window_days)
    end_exclusive = target + timedelta(days=window_days + 1)
    end_inclusive = end_exclusive - timedelta(days=1)
    return start.isoformat(), end_inclusive.isoformat(), end_exclusive.isoformat()


def empty_valid_image(aoi: ee.Geometry) -> ee.Image:
    return ee.Image.constant(0).rename("valid").toByte().clip(aoi)


def any_valid_mask(collection: ee.ImageCollection, valid_fn, aoi: ee.Geometry) -> ee.Image:
    valid_collection = ee.ImageCollection(
        collection.map(lambda image: valid_fn(ee.Image(image)).unmask(0))
    ).merge(ee.ImageCollection([empty_valid_image(aoi)]))
    return valid_collection.sum().gt(0).rename("valid_any").clip(aoi)


def area_stats(mask_dict: dict[str, ee.Image], aoi: ee.Geometry) -> dict[str, float]:
    area_bands = []
    for name, mask in mask_dict.items():
        area_bands.append(ee.Image.pixelArea().rename(name).updateMask(mask))
    stats = ee.Image.cat(area_bands).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=AREA_SCALE_M,
        maxPixels=1e13,
        tileScale=REDUCE_TILE_SCALE,
    ).getInfo()
    return {name: (stats.get(name) or 0) / 1_000_000 for name in mask_dict}


def pct(area_km2: float, denominator_km2: float) -> float:
    if denominator_km2 <= 0:
        return 0.0
    return 100.0 * area_km2 / denominator_km2


def build_window_sources(aoi: ee.Geometry, target_date: str, window_days: int) -> dict:
    start, end_inclusive, end_exclusive = window_bounds(target_date, window_days)
    dw_collection = dynamic_world_collection(aoi, start, end_exclusive)
    hls_collection = opera_dswx_hls_collection(
        aoi,
        start,
        end_exclusive,
        include_sentinel2=INCLUDE_HLS_SENTINEL2,
    )
    s1_collection = opera_dswx_s1_collection(aoi, start, end_exclusive)

    dw_valid = any_valid_mask(dw_collection, dw_valid_observation, aoi).unmask(0).eq(1)
    hls_valid = any_valid_mask(hls_collection, hls_valid_observation, aoi).unmask(0).eq(1)
    s1_valid = any_valid_mask(s1_collection, s1_valid_observation, aoi).unmask(0).eq(1)

    dw_gap = dw_valid.Not()
    hls_used = dw_gap.And(hls_valid)
    after_hls_gap = dw_gap.And(hls_valid.Not())
    s1_used = after_hls_gap.And(s1_valid)
    final_valid = dw_valid.Or(hls_used).Or(s1_used)
    remaining_gap = final_valid.Not()

    return {
        "window_start": start,
        "window_end_inclusive": end_inclusive,
        "window_end_exclusive": end_exclusive,
        "collections": {"dw": dw_collection, "hls": hls_collection, "s1": s1_collection},
        "masks": {
            "dw_valid": dw_valid,
            "hls_valid": hls_valid,
            "s1_valid": s1_valid,
            "dw_gap": dw_gap,
            "hls_used": hls_used,
            "after_hls_gap": after_hls_gap,
            "s1_used": s1_used,
            "final_valid": final_valid,
            "remaining_gap": remaining_gap,
        },
    }


def scan_cell_window(cell_record: dict, target_date: str, window_days: int) -> dict:
    aoi = effective_cell_geometry(cell_record["geometry"])
    area_km2 = cell_record["aoi_area_km2"]
    if area_km2 is None:
        area_km2 = geometry_area_km2(aoi)
    sources = build_window_sources(aoi, target_date, window_days)
    masks = sources["masks"]
    areas = area_stats(masks, aoi)

    if INCLUDE_IMAGE_COUNTS:
        counts = ee.Dictionary({
            "dw_image_count": sources["collections"]["dw"].size(),
            "hls_image_count": sources["collections"]["hls"].size(),
            "s1_image_count": sources["collections"]["s1"].size(),
        }).getInfo()
    else:
        counts = {"dw_image_count": None, "hls_image_count": None, "s1_image_count": None}

    dw_gap_area = areas["dw_gap"]
    after_hls_gap_area = areas["after_hls_gap"]

    return {
        "cell_id": cell_record["cell_id"],
        "utm_zone": cell_record["utm_zone"],
        "utm_row": cell_record["utm_row"],
        "row_order": cell_record["row_order"],
        "lat_min": cell_record["lat_min"],
        "lat_max": cell_record["lat_max"],
        "lat_center": cell_record["lat_center"],
        "lon_min": cell_record["lon_min"],
        "lon_max": cell_record["lon_max"],
        "aoi_area_km2": area_km2,
        "target_date": target_date,
        "window_days": window_days,
        "window_start": sources["window_start"],
        "window_end_inclusive": sources["window_end_inclusive"],
        "window_end_exclusive": sources["window_end_exclusive"],
        **counts,
        "dw_valid_pct": pct(areas["dw_valid"], area_km2),
        "hls_valid_pct": pct(areas["hls_valid"], area_km2),
        "s1_valid_pct": pct(areas["s1_valid"], area_km2),
        "hls_used_pct_aoi": pct(areas["hls_used"], area_km2),
        "s1_used_pct_aoi": pct(areas["s1_used"], area_km2),
        "final_valid_pct": pct(areas["final_valid"], area_km2),
        "remaining_gap_pct": pct(areas["remaining_gap"], area_km2),
        "hls_fills_dw_gap_pct": pct(areas["hls_used"], dw_gap_area),
        "s1_fills_after_hls_gap_pct": pct(areas["s1_used"], after_hls_gap_area),
        "dw_valid_km2": areas["dw_valid"],
        "hls_used_km2": areas["hls_used"],
        "s1_used_km2": areas["s1_used"],
        "final_valid_km2": areas["final_valid"],
        "remaining_gap_km2": areas["remaining_gap"],
    }

## 8. Run Window Scan

Results are saved incrementally to `utm_window_scan_results.csv`. If the run stops, rerun with `REUSE_EXISTING_WINDOW_SCAN = True`.

In [8]:
window_days_list = list(range(0, MAX_WINDOW_DAYS + 1, WINDOW_STEP_DAYS))
window_csv = CSV_DIR / "utm_window_scan_results.csv"

existing_rows = []
completed_keys = set()
if REUSE_EXISTING_WINDOW_SCAN and window_csv.exists():
    existing_df = pd.read_csv(window_csv)
    required_cols = {"cell_id", "target_date", "window_days"}
    if required_cols.issubset(existing_df.columns):
        existing_rows = existing_df.to_dict("records")
        completed_keys = {
            (str(row["cell_id"]), str(row["target_date"]), int(row["window_days"]))
            for row in existing_rows
        }
        print(f"Loaded {len(existing_rows)} existing rows from {window_csv}")

scan_plan = [
    (cell_record, target_date, window_days)
    for cell_record in cell_records
    for target_date in target_dates
    for window_days in window_days_list
]
remaining_plan = [
    item for item in scan_plan
    if (str(item[0]["cell_id"]), str(item[1]), int(item[2])) not in completed_keys
]

print(
    f"Scan plan: {len(cell_records)} cells x {len(target_dates)} dates x "
    f"{len(window_days_list)} windows = {len(scan_plan)} rows"
)
print(f"Remaining rows to compute: {len(remaining_plan)}")

rows = list(existing_rows)
for i, (cell_record, target_date, window_days) in enumerate(remaining_plan, start=1):
    print(f"[{i}/{len(remaining_plan)}] cell={cell_record['cell_id']}, target={target_date}, window=+-{window_days}")
    rows.append(scan_cell_window(cell_record, target_date, window_days))
    pd.DataFrame(rows).sort_values(["cell_id", "target_date", "window_days"]).to_csv(window_csv, index=False)

window_df = pd.DataFrame(rows).sort_values(["cell_id", "target_date", "window_days"]).reset_index(drop=True)
window_df.to_csv(window_csv, index=False)

print("Saved:", window_csv)
display(window_df.head(20))

Scan plan: 19 cells x 1 dates x 31 windows = 589 rows
Remaining rows to compute: 589
[1/589] cell=Z34C, target=2025-05-01, window=+-0
[2/589] cell=Z34C, target=2025-05-01, window=+-1
[3/589] cell=Z34C, target=2025-05-01, window=+-2
[4/589] cell=Z34C, target=2025-05-01, window=+-3
[5/589] cell=Z34C, target=2025-05-01, window=+-4
[6/589] cell=Z34C, target=2025-05-01, window=+-5
[7/589] cell=Z34C, target=2025-05-01, window=+-6
[8/589] cell=Z34C, target=2025-05-01, window=+-7
[9/589] cell=Z34C, target=2025-05-01, window=+-8
[10/589] cell=Z34C, target=2025-05-01, window=+-9
[11/589] cell=Z34C, target=2025-05-01, window=+-10
[12/589] cell=Z34C, target=2025-05-01, window=+-11
[13/589] cell=Z34C, target=2025-05-01, window=+-12
[14/589] cell=Z34C, target=2025-05-01, window=+-13
[15/589] cell=Z34C, target=2025-05-01, window=+-14
[16/589] cell=Z34C, target=2025-05-01, window=+-15
[17/589] cell=Z34C, target=2025-05-01, window=+-16
[18/589] cell=Z34C, target=2025-05-01, window=+-17
[19/589] cell=Z3

,cell_id,utm_zone,utm_row,row_order,lat_min,lat_max,lat_center,lon_min,lon_max,aoi_area_km2,...,s1_used_pct_aoi,final_valid_pct,remaining_gap_pct,hls_fills_dw_gap_pct,s1_fills_after_hls_gap_pct,dw_valid_km2,hls_used_km2,s1_used_km2,final_valid_km2,remaining_gap_km2
0,Z34C,34,C,0.0,-80.0,-72.0,-76.0,18.0,24.0,143454.618376,...,0.0,0.0,100.821818,0.0,0.0,0.0,0.0,0.0,0.0,144633.553816
1,Z34C,34,C,0.0,-80.0,-72.0,-76.0,18.0,24.0,143454.618376,...,0.0,0.0,100.821818,0.0,0.0,0.0,0.0,0.0,0.0,144633.553816
2,Z34C,34,C,0.0,-80.0,-72.0,-76.0,18.0,24.0,143454.618376,...,0.0,0.0,100.821818,0.0,0.0,0.0,0.0,0.0,0.0,144633.553816
3,Z34C,34,C,0.0,-80.0,-72.0,-76.0,18.0,24.0,143454.618376,...,0.0,0.0,100.821818,0.0,0.0,0.0,0.0,0.0,0.0,144633.553816
4,Z34C,34,C,0.0,-80.0,-72.0,-76.0,18.0,24.0,143454.618376,...,0.0,0.0,100.821818,0.0,0.0,0.0,0.0,0.0,0.0,144633.553816
5,Z34C,34,C,0.0,-80.0,-72.0,-76.0,18.0,24.0,143454.618376,...,0.0,0.0,100.821818,0.0,0.0,0.0,0.0,0.0,0.0,144633.553816
6,Z34C,34,C,0.0,-80.0,-72.0,-76.0,18.0,24.0,143454.618376,...,0.0,0.0,100.821818,0.0,0.0,0.0,0.0,0.0,0.0,144633.553816
7,Z34C,34,C,0.0,-80.0,-72.0,-76.0,18.0,24.0,143454.618376,...,0.0,0.0,100.821818,0.0,0.0,0.0,0.0,0.0,0.0,144633.553816
8,Z34C,34,C,0.0,-80.0,-72.0,-76.0,18.0,24.0,143454.618376,...,0.0,0.0,100.821818,0.0,0.0,0.0,0.0,0.0,0.0,144633.553816
9,Z34C,34,C,0.0,-80.0,-72.0,-76.0,18.0,24.0,143454.618376,...,0.0,0.0,100.821818,0.0,0.0,0.0,0.0,0.0,0.0,144633.553816


## 9. Build Summary Tables

The summary tables separate the fixed primary window from the selected optimal window for each cell/date.

In [9]:
if "window_df" not in globals():
    if window_csv.exists():
        window_df = pd.read_csv(window_csv)
    else:
        raise RuntimeError("Run Section 8 first, or point OUTPUT_DIR to an existing scan folder.")


def select_optimal_window(group: pd.DataFrame) -> pd.Series:
    ordered = group.sort_values(["window_days", "remaining_gap_pct"])
    eligible = ordered[ordered["final_valid_pct"] >= COVERAGE_TARGET_PCT]
    if not eligible.empty:
        row = eligible.iloc[0].copy()
        row["meets_target"] = True
        return row
    best = group.sort_values(["final_valid_pct", "window_days"], ascending=[False, True]).iloc[0].copy()
    best["meets_target"] = False
    return best


optimal_rows = []
for (cell_id, target_date), group in window_df.groupby(["cell_id", "target_date"], sort=True):
    row = select_optimal_window(group).copy()
    row["cell_id"] = cell_id
    row["target_date"] = target_date
    optimal_rows.append(row)
optimal_df = pd.DataFrame(optimal_rows).reset_index(drop=True)

primary_df = window_df[window_df["window_days"] == PRIMARY_WINDOW_DAYS].copy().reset_index(drop=True)
if primary_df.empty:
    raise RuntimeError(f"No rows found for PRIMARY_WINDOW_DAYS={PRIMARY_WINDOW_DAYS}.")

per_cell_date = primary_df.add_prefix("primary_").merge(
    optimal_df.add_prefix("optimal_"),
    left_on=["primary_cell_id", "primary_target_date"],
    right_on=["optimal_cell_id", "optimal_target_date"],
    how="left",
)
per_cell_date["cell_id"] = per_cell_date["primary_cell_id"]
per_cell_date["target_date"] = per_cell_date["primary_target_date"]
per_cell_date["utm_zone"] = per_cell_date["primary_utm_zone"]
per_cell_date["utm_row"] = per_cell_date["primary_utm_row"]
per_cell_date["lat_center"] = per_cell_date["primary_lat_center"]
per_cell_date["primary_meets_target"] = per_cell_date["primary_final_valid_pct"] >= COVERAGE_TARGET_PCT
per_cell_date["adaptive_window_needed"] = per_cell_date["optimal_window_days"] > PRIMARY_WINDOW_DAYS


def q90(series: pd.Series) -> float:
    return float(series.quantile(0.9))


def ceil_q90(series: pd.Series) -> int:
    return int(math.ceil(q90(series)))


cell_summary = per_cell_date.groupby(
    ["cell_id", "utm_zone", "utm_row", "primary_row_order", "primary_lat_min", "primary_lat_max", "lat_center"],
    dropna=False,
).agg(
    target_date_count=("target_date", "nunique"),
    aoi_area_km2=("primary_aoi_area_km2", "first"),
    primary_mean_final_valid_pct=("primary_final_valid_pct", "mean"),
    primary_min_final_valid_pct=("primary_final_valid_pct", "min"),
    primary_mean_remaining_gap_pct=("primary_remaining_gap_pct", "mean"),
    primary_target_met_pct=("primary_meets_target", lambda s: 100.0 * s.mean()),
    primary_mean_dw_valid_pct=("primary_dw_valid_pct", "mean"),
    primary_mean_hls_used_pct_aoi=("primary_hls_used_pct_aoi", "mean"),
    primary_mean_s1_used_pct_aoi=("primary_s1_used_pct_aoi", "mean"),
    optimal_median_window_days=("optimal_window_days", "median"),
    optimal_p90_window_days=("optimal_window_days", q90),
    recommended_window_days_p90=("optimal_window_days", ceil_q90),
    optimal_max_window_days=("optimal_window_days", "max"),
    optimal_target_met_pct=("optimal_meets_target", lambda s: 100.0 * s.mean()),
).reset_index().rename(columns={
    "primary_row_order": "row_order",
    "primary_lat_min": "lat_min",
    "primary_lat_max": "lat_max",
})

row_summary = per_cell_date.groupby(["utm_row", "primary_row_order"], dropna=False).agg(
    cell_count=("cell_id", "nunique"),
    target_date_count=("target_date", "nunique"),
    mean_lat_center=("lat_center", "mean"),
    primary_mean_final_valid_pct=("primary_final_valid_pct", "mean"),
    primary_min_final_valid_pct=("primary_final_valid_pct", "min"),
    primary_target_met_pct=("primary_meets_target", lambda s: 100.0 * s.mean()),
    primary_mean_dw_valid_pct=("primary_dw_valid_pct", "mean"),
    primary_mean_hls_used_pct_aoi=("primary_hls_used_pct_aoi", "mean"),
    primary_mean_s1_used_pct_aoi=("primary_s1_used_pct_aoi", "mean"),
    primary_mean_remaining_gap_pct=("primary_remaining_gap_pct", "mean"),
    optimal_median_window_days=("optimal_window_days", "median"),
    optimal_p90_window_days=("optimal_window_days", q90),
    recommended_window_days_p90=("optimal_window_days", ceil_q90),
    optimal_max_window_days=("optimal_window_days", "max"),
    optimal_target_met_pct=("optimal_meets_target", lambda s: 100.0 * s.mean()),
).reset_index().rename(columns={"primary_row_order": "row_order"})
row_summary = row_summary.sort_values("row_order").reset_index(drop=True)

row_date_summary = per_cell_date.groupby(["utm_row", "primary_row_order", "target_date"], dropna=False).agg(
    cell_count=("cell_id", "nunique"),
    primary_mean_final_valid_pct=("primary_final_valid_pct", "mean"),
    primary_target_met_pct=("primary_meets_target", lambda s: 100.0 * s.mean()),
    optimal_median_window_days=("optimal_window_days", "median"),
    recommended_window_days_p90=("optimal_window_days", ceil_q90),
    optimal_target_met_pct=("optimal_meets_target", lambda s: 100.0 * s.mean()),
).reset_index().rename(columns={"primary_row_order": "row_order"})

optimal_csv = CSV_DIR / "utm_optimal_windows.csv"
primary_csv = CSV_DIR / "utm_fixed_primary_windows.csv"
per_cell_date_csv = CSV_DIR / "utm_per_cell_date_summary.csv"
cell_summary_csv = CSV_DIR / "utm_cell_summary.csv"
row_summary_csv = CSV_DIR / "utm_row_summary.csv"
row_date_summary_csv = CSV_DIR / "utm_row_date_summary.csv"

optimal_df.to_csv(optimal_csv, index=False)
primary_df.to_csv(primary_csv, index=False)
per_cell_date.to_csv(per_cell_date_csv, index=False)
cell_summary.to_csv(cell_summary_csv, index=False)
row_summary.to_csv(row_summary_csv, index=False)
row_date_summary.to_csv(row_date_summary_csv, index=False)

print("Saved:", optimal_csv)
print("Saved:", primary_csv)
print("Saved:", per_cell_date_csv)
print("Saved:", cell_summary_csv)
print("Saved:", row_summary_csv)
print("Saved:", row_date_summary_csv)
display(row_summary)

Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\utm_zone_row_window_preanalysis\utm_zone_34_20260703_120513\csv\utm_optimal_windows.csv
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\utm_zone_row_window_preanalysis\utm_zone_34_20260703_120513\csv\utm_fixed_primary_windows.csv
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\utm_zone_row_window_preanalysis\utm_zone_34_20260703_120513\csv\utm_per_cell_date_summary.csv
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\utm_zone_row_window_preanalysis\utm_zone_34_20260703_120513\csv\utm_cell_summary.csv
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\utm_zone_row_window_preanalysis\utm_zone_34_20260703_120513\csv\utm_row_summary.csv
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\utm_zone_row_window_preanalysis\utm_zone_34_20260703_120513\csv\utm_row_date_summary.csv


,utm_row,row_order,cell_count,target_date_count,mean_lat_center,primary_mean_final_valid_pct,primary_min_final_valid_pct,primary_target_met_pct,primary_mean_dw_valid_pct,primary_mean_hls_used_pct_aoi,primary_mean_s1_used_pct_aoi,primary_mean_remaining_gap_pct,optimal_median_window_days,optimal_p90_window_days,recommended_window_days_p90,optimal_max_window_days,optimal_target_met_pct
0,C,0.0,1,1,-76.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,100.821818,0.0,0.0,0,0,0.0
1,D,1.0,1,1,-68.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,100.707175,27.0,27.0,27,27,0.0
2,E,2.0,1,1,-60.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,100.558420,0.0,0.0,0,0,0.0
3,F,3.0,1,1,-52.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,100.384045,0.0,0.0,0,0,0.0
4,G,4.0,1,1,-44.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,100.197561,0.0,0.0,0,0,0.0
5,H,5.0,1,1,-36.0,53.319503,53.319503,0.0,33.156044,5.584152,14.579306,46.693966,29.0,29.0,29,29,0.0
6,J,6.0,1,1,-28.0,99.810882,99.810882,100.0,96.620361,3.166818,0.023703,0.039393,5.0,5.0,5,5,100.0
7,K,7.0,1,1,-20.0,99.328291,99.328291,100.0,97.303877,1.977674,0.046740,0.379632,4.0,4.0,4,4,100.0
8,L,8.0,1,1,-12.0,98.876348,98.876348,0.0,86.527301,9.470189,2.878858,0.733135,6.0,6.0,6,6,100.0
9,M,9.0,1,1,-4.0,92.477258,92.477258,0.0,40.196219,28.931546,23.349492,7.081085,6.0,6.0,6,6,100.0


## 10. Save Spatial Summary GeoJSON

This joins cell-level results back to the selected UTM grid so you can inspect recommended windows spatially in GIS.

In [10]:
cell_summary_spatial = selected_grid.merge(
    cell_summary,
    on=["cell_id", "utm_zone", "utm_row"],
    how="inner",
    suffixes=("", "_summary"),
)
spatial_summary_geojson = OUTPUT_DIR / "utm_cell_summary.geojson"
cell_summary_spatial.to_file(spatial_summary_geojson, driver="GeoJSON")
print("Saved:", spatial_summary_geojson)
display(cell_summary_spatial.drop(columns="geometry").head(20))

Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\utm_zone_row_window_preanalysis\utm_zone_34_20260703_120513\utm_cell_summary.geojson


,ZONE,ROW_,WEST_VALUE,CM_VALUE,EAST_VALUE,utm_zone,utm_row,cell_id,lon_min,lat_min,...,primary_mean_remaining_gap_pct,primary_target_met_pct,primary_mean_dw_valid_pct,primary_mean_hls_used_pct_aoi,primary_mean_s1_used_pct_aoi,optimal_median_window_days,optimal_p90_window_days,recommended_window_days_p90,optimal_max_window_days,optimal_target_met_pct
0,34.0,C,18E,21E,24E,34,C,Z34C,18.0,-8.000000e+01,...,100.821818,0.0,0.000000,0.000000,0.000000,0.0,0.0,0,0,0.0
1,34.0,D,18E,21E,24E,34,D,Z34D,18.0,-7.200000e+01,...,100.707175,0.0,0.000000,0.000000,0.000000,27.0,27.0,27,27,0.0
2,34.0,E,18E,21E,24E,34,E,Z34E,18.0,-6.400000e+01,...,100.558420,0.0,0.000000,0.000000,0.000000,0.0,0.0,0,0,0.0
3,34.0,F,18E,21E,24E,34,F,Z34F,18.0,-5.600000e+01,...,100.384045,0.0,0.000000,0.000000,0.000000,0.0,0.0,0,0,0.0
4,34.0,G,18E,21E,24E,34,G,Z34G,18.0,-4.800000e+01,...,100.197561,0.0,0.000000,0.000000,0.000000,0.0,0.0,0,0,0.0
5,34.0,H,18E,21E,24E,34,H,Z34H,18.0,-4.000000e+01,...,46.693966,0.0,33.156044,5.584152,14.579306,29.0,29.0,29,29,0.0
6,34.0,J,18E,21E,24E,34,J,Z34J,18.0,-3.200000e+01,...,0.039393,100.0,96.620361,3.166818,0.023703,5.0,5.0,5,5,100.0
7,34.0,K,18E,21E,24E,34,K,Z34K,18.0,-2.400000e+01,...,0.379632,100.0,97.303877,1.977674,0.046740,4.0,4.0,4,4,100.0
8,34.0,L,18E,21E,24E,34,L,Z34L,18.0,-1.600000e+01,...,0.733135,0.0,86.527301,9.470189,2.878858,6.0,6.0,6,6,100.0
9,34.0,M,18E,21E,24E,34,M,Z34M,18.0,-8.000000e+00,...,7.081085,0.0,40.196219,28.931546,23.349492,6.0,6.0,6,6,100.0


## 11. Save PNG Figures

Color meanings:

- Blue: Dynamic World valid coverage.
- Orange: OPERA HLS Landsat contribution where DW is invalid.
- Purple: OPERA S1 contribution where DW and HLS are invalid.
- Red: remaining gap.
- Dark gray/black: final valid coverage or selected window line.

In [11]:
from PIL import Image, ImageDraw, ImageFont

COLORS = {
    "dw": "#419bdf",
    "hls": "#e49635",
    "s1": "#7a87c6",
    "gap": "#d73027",
    "final": "#111111",
    "target": "#666666",
    "grid": "#dddddd",
    "axis": "#555555",
    "text": "#222222",
    "muted": "#666666",
    "bg": "#ffffff",
    "heat_low": "#1a9850",
    "heat_mid": "#fee08b",
    "heat_high": "#d73027",
}


def rgb(value: str) -> tuple[int, int, int]:
    value = value.lstrip("#")
    return tuple(int(value[i:i + 2], 16) for i in (0, 2, 4))


def blend(c1: str, c2: str, t: float) -> tuple[int, int, int]:
    a = rgb(c1)
    b = rgb(c2)
    t = max(0.0, min(1.0, float(t)))
    return tuple(round(a[i] + (b[i] - a[i]) * t) for i in range(3))


def heat_color(value: float, vmin: float, vmax: float) -> tuple[int, int, int]:
    if vmax <= vmin:
        return rgb(COLORS["heat_low"])
    t = (float(value) - vmin) / (vmax - vmin)
    if t <= 0.5:
        return blend(COLORS["heat_low"], COLORS["heat_mid"], t / 0.5)
    return blend(COLORS["heat_mid"], COLORS["heat_high"], (t - 0.5) / 0.5)


def get_font(size: int, bold: bool = False):
    candidates = [
        "C:/Windows/Fonts/arialbd.ttf" if bold else "C:/Windows/Fonts/arial.ttf",
        "C:/Windows/Fonts/calibrib.ttf" if bold else "C:/Windows/Fonts/calibri.ttf",
    ]
    for candidate in candidates:
        try:
            return ImageFont.truetype(candidate, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_TITLE = get_font(20, True)
FONT_LABEL = get_font(12)
FONT_SMALL = get_font(10)


def text_w(draw, text, font=FONT_SMALL):
    box = draw.textbbox((0, 0), str(text), font=font)
    return box[2] - box[0]


def draw_legend(draw, x, y, entries):
    cursor = x
    for label, color in entries:
        draw.rectangle((cursor, y + 3, cursor + 12, y + 15), fill=rgb(color))
        draw.text((cursor + 17, y), label, fill=rgb(COLORS["text"]), font=FONT_LABEL)
        cursor += 30 + text_w(draw, label, FONT_LABEL)


def save_row_stacked_contribution(df: pd.DataFrame, path: Path):
    data = df.sort_values("row_order").reset_index(drop=True)
    n = len(data)
    width = max(950, 150 + n * 42)
    height = 540
    left, right, top, bottom = 70, 30, 90, 85
    plot_w = width - left - right
    plot_h = height - top - bottom
    image = Image.new("RGB", (width, height), rgb(COLORS["bg"]))
    draw = ImageDraw.Draw(image)
    draw.text((left, 22), f"Mean source contribution by UTM row, fixed +/-{PRIMARY_WINDOW_DAYS} days", fill=rgb(COLORS["text"]), font=FONT_TITLE)
    draw_legend(draw, left, 54, [("Dynamic World", COLORS["dw"]), ("HLS fill", COLORS["hls"]), ("S1 fill", COLORS["s1"]), ("Remaining gap", COLORS["gap"])])

    def sy(value):
        return top + (100.0 - float(value)) / 100.0 * plot_h

    for tick in [0, 25, 50, 75, 100]:
        y = sy(tick)
        draw.line((left, y, left + plot_w, y), fill=rgb(COLORS["grid"]))
        draw.text((18, y - 7), f"{tick}%", fill=rgb(COLORS["muted"]), font=FONT_SMALL)
    draw.line((left, top, left, top + plot_h), fill=rgb(COLORS["axis"]))
    draw.line((left, top + plot_h, left + plot_w, top + plot_h), fill=rgb(COLORS["axis"]))

    bar_w = max(10, min(30, plot_w / max(n, 1) * 0.65))
    step = (plot_w - bar_w) / max(n - 1, 1)
    for idx, row in data.iterrows():
        x0 = left + idx * step
        x1 = x0 + bar_w
        y_base = top + plot_h
        cumulative = 0.0
        for column, color in [
            ("primary_mean_dw_valid_pct", COLORS["dw"]),
            ("primary_mean_hls_used_pct_aoi", COLORS["hls"]),
            ("primary_mean_s1_used_pct_aoi", COLORS["s1"]),
            ("primary_mean_remaining_gap_pct", COLORS["gap"]),
        ]:
            cumulative += float(row[column])
            y_top = sy(cumulative)
            draw.rectangle((x0, y_top, x1, y_base), fill=rgb(color))
            y_base = y_top
        label = str(row["utm_row"])
        draw.text((x0 + bar_w / 2 - text_w(draw, label) / 2, top + plot_h + 10), label, fill=rgb(COLORS["muted"]), font=FONT_SMALL)
    image.save(path)


def save_row_window_plot(df: pd.DataFrame, path: Path):
    data = df.sort_values("row_order").reset_index(drop=True)
    n = len(data)
    y_max = max(MAX_WINDOW_DAYS, int(data["recommended_window_days_p90"].max()))
    width = max(950, 150 + n * 42)
    height = 480
    left, right, top, bottom = 70, 30, 80, 80
    plot_w = width - left - right
    plot_h = height - top - bottom
    image = Image.new("RGB", (width, height), rgb(COLORS["bg"]))
    draw = ImageDraw.Draw(image)
    draw.text((left, 22), "Recommended window by UTM row", fill=rgb(COLORS["text"]), font=FONT_TITLE)

    def sy(value):
        return top + (float(y_max) - float(value)) / max(float(y_max), 1.0) * plot_h

    for tick in range(0, y_max + 1, max(1, math.ceil(y_max / 5))):
        y = sy(tick)
        draw.line((left, y, left + plot_w, y), fill=rgb(COLORS["grid"]))
        draw.text((25, y - 7), f"{tick} d", fill=rgb(COLORS["muted"]), font=FONT_SMALL)
    primary_y = sy(PRIMARY_WINDOW_DAYS)
    draw.line((left, primary_y, left + plot_w, primary_y), fill=rgb(COLORS["target"]))
    draw.text((left + plot_w - 95, primary_y - 18), f"primary +/-{PRIMARY_WINDOW_DAYS}", fill=rgb(COLORS["muted"]), font=FONT_SMALL)
    draw.line((left, top, left, top + plot_h), fill=rgb(COLORS["axis"]))
    draw.line((left, top + plot_h, left + plot_w, top + plot_h), fill=rgb(COLORS["axis"]))

    bar_w = max(10, min(30, plot_w / max(n, 1) * 0.65))
    step = (plot_w - bar_w) / max(n - 1, 1)
    for idx, row in data.iterrows():
        x = left + idx * step + bar_w / 2
        y = sy(row["recommended_window_days_p90"])
        draw.rectangle((x - bar_w / 2, y, x + bar_w / 2, top + plot_h), fill=rgb(COLORS["hls"]))
        label = str(row["utm_row"])
        draw.text((x - text_w(draw, label) / 2, top + plot_h + 10), label, fill=rgb(COLORS["muted"]), font=FONT_SMALL)
    image.save(path)


def save_row_date_heatmap(df: pd.DataFrame, path: Path):
    data = df.copy()
    rows = sorted(data["utm_row"].dropna().unique().tolist(), key=lambda r: ROW_ORDER.get(r, 999))
    dates = sorted(data["target_date"].dropna().astype(str).unique().tolist())
    cell_w = 22
    cell_h = 22
    left = 70
    top = 80
    width = max(900, left + len(dates) * cell_w + 50)
    height = top + len(rows) * cell_h + 90
    image = Image.new("RGB", (width, height), rgb(COLORS["bg"]))
    draw = ImageDraw.Draw(image)
    draw.text((left, 22), "Median optimal window days by UTM row and target date", fill=rgb(COLORS["text"]), font=FONT_TITLE)
    value_map = {
        (row["utm_row"], str(row["target_date"])): row["optimal_median_window_days"]
        for _, row in data.iterrows()
    }
    for r_idx, row_name in enumerate(rows):
        y = top + r_idx * cell_h
        draw.text((35, y + 5), str(row_name), fill=rgb(COLORS["muted"]), font=FONT_SMALL)
        for d_idx, target_date in enumerate(dates):
            x = left + d_idx * cell_w
            value = value_map.get((row_name, target_date))
            fill = rgb("#f0f0f0") if pd.isna(value) else heat_color(value, 0, MAX_WINDOW_DAYS)
            draw.rectangle((x, y, x + cell_w - 1, y + cell_h - 1), fill=fill)
    tick_every = max(1, round(len(dates) / 12))
    for d_idx, target_date in enumerate(dates):
        if d_idx % tick_every == 0 or d_idx == len(dates) - 1:
            x = left + d_idx * cell_w
            label = target_date[5:]
            draw.text((x - 4, top + len(rows) * cell_h + 8), label, fill=rgb(COLORS["muted"]), font=FONT_SMALL)
    draw.text((left, height - 28), f"Green=0 days, yellow=mid, red={MAX_WINDOW_DAYS} days", fill=rgb(COLORS["muted"]), font=FONT_SMALL)
    image.save(path)


figure_manifest = []


def register(path: Path, description: str):
    figure_manifest.append({"path": str(path), "description": description})
    print("Saved:", path)


row_contrib_png = FIGURE_DIR / "row_mean_source_contribution_fixed_window.png"
save_row_stacked_contribution(row_summary, row_contrib_png)
register(row_contrib_png, "Mean fixed-window source contribution by UTM row.")

row_window_png = FIGURE_DIR / "row_recommended_window_days.png"
save_row_window_plot(row_summary, row_window_png)
register(row_window_png, "P90 recommended +/- day window by UTM row.")

row_date_heatmap_png = FIGURE_DIR / "row_date_optimal_window_heatmap.png"
save_row_date_heatmap(row_date_summary, row_date_heatmap_png)
register(row_date_heatmap_png, "Median optimal window days by UTM row and target date.")

figure_manifest_csv = CSV_DIR / "figure_manifest.csv"
pd.DataFrame(figure_manifest).to_csv(figure_manifest_csv, index=False)
print("Saved:", figure_manifest_csv)

Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\utm_zone_row_window_preanalysis\utm_zone_34_20260703_120513\figures\row_mean_source_contribution_fixed_window.png
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\utm_zone_row_window_preanalysis\utm_zone_34_20260703_120513\figures\row_recommended_window_days.png
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\utm_zone_row_window_preanalysis\utm_zone_34_20260703_120513\figures\row_date_optimal_window_heatmap.png
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\utm_zone_row_window_preanalysis\utm_zone_34_20260703_120513\csv\figure_manifest.csv


## 12. Save Parameters And Output Index

In [12]:
parameters = {
    "utm_grid_zip": UTM_GRID_ZIP,
    "analysis_mode": ANALYSIS_MODE,
    "utm_zone": UTM_ZONE,
    "utm_rows": UTM_ROWS,
    "include_polar_rows": INCLUDE_POLAR_ROWS,
    "continent_name": CONTINENT_NAME,
    "continent_collection": CONTINENT_COLLECTION,
    "continent_property": CONTINENT_PROPERTY,
    "continent_value": CONTINENT_VALUE,
    "clip_cells_to_continent": CLIP_CELLS_TO_CONTINENT,
    "continent_prefilter_bbox": CONTINENT_PREFILTER_BBOX,
    "min_cell_area_km2": MIN_CELL_AREA_KM2,
    "start_date": START_DATE,
    "end_date_exclusive": END_DATE,
    "target_date_mode": TARGET_DATE_MODE,
    "manual_target_dates": MANUAL_TARGET_DATES,
    "primary_window_days": PRIMARY_WINDOW_DAYS,
    "max_reference_dates": MAX_REFERENCE_DATES,
    "max_window_days": MAX_WINDOW_DAYS,
    "window_step_days": WINDOW_STEP_DAYS,
    "coverage_target_pct": COVERAGE_TARGET_PCT,
    "area_scale_m": AREA_SCALE_M,
    "reduce_tile_scale": REDUCE_TILE_SCALE,
    "include_image_counts": INCLUDE_IMAGE_COUNTS,
    "include_hls_sentinel2": INCLUDE_HLS_SENTINEL2,
    "output_dir": str(OUTPUT_DIR),
}
parameters_json = OUTPUT_DIR / "parameters.json"
with parameters_json.open("w", encoding="utf-8") as f:
    json.dump(parameters, f, indent=2)

output_index = pd.DataFrame([
    {"file": str(CSV_DIR / "selected_cell_metadata.csv"), "description": "Selected UTM cells and effective AOI area."},
    {"file": str(CSV_DIR / "target_dates.csv"), "description": "Reference dates scanned."},
    {"file": str(CSV_DIR / "utm_window_scan_results.csv"), "description": "Detailed cell-date-window scan results."},
    {"file": str(CSV_DIR / "utm_fixed_primary_windows.csv"), "description": f"Fixed +/-{PRIMARY_WINDOW_DAYS} window rows."},
    {"file": str(CSV_DIR / "utm_optimal_windows.csv"), "description": "Minimum window reaching target, or best available window."},
    {"file": str(CSV_DIR / "utm_per_cell_date_summary.csv"), "description": "Merged fixed-vs-optimal per cell and date."},
    {"file": str(CSV_DIR / "utm_cell_summary.csv"), "description": "Aggregated cell-level summary."},
    {"file": str(CSV_DIR / "utm_row_summary.csv"), "description": "Aggregated UTM latitude-row summary."},
    {"file": str(CSV_DIR / "utm_row_date_summary.csv"), "description": "UTM row by date summary."},
    {"file": str(OUTPUT_DIR / "selected_utm_cells.geojson"), "description": "Selected source UTM cells."},
    {"file": str(OUTPUT_DIR / "utm_cell_summary.geojson"), "description": "UTM cells with summary metrics."},
    {"file": str(CSV_DIR / "figure_manifest.csv"), "description": "Saved PNG figure index."},
    {"file": str(parameters_json), "description": "Run parameters."},
])
output_index_csv = CSV_DIR / "output_index.csv"
output_index.to_csv(output_index_csv, index=False)

print("Saved:", parameters_json)
print("Saved:", output_index_csv)
display(output_index)

Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\utm_zone_row_window_preanalysis\utm_zone_34_20260703_120513\parameters.json
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\utm_zone_row_window_preanalysis\utm_zone_34_20260703_120513\csv\output_index.csv


,file,description
0,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,Selected UTM cells and effective AOI area.
1,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,Reference dates scanned.
2,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,Detailed cell-date-window scan results.
3,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,Fixed +/-5 window rows.
4,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,"Minimum window reaching target, or best availa..."
5,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,Merged fixed-vs-optimal per cell and date.
6,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,Aggregated cell-level summary.
7,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,Aggregated UTM latitude-row summary.
8,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,UTM row by date summary.
9,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,Selected source UTM cells.


## Takeaways

After a run, start with:

- `csv/utm_row_summary.csv`: recommended window by UTM latitude row.
- `csv/utm_cell_summary.csv`: recommended window by individual UTM cell.
- `csv/utm_per_cell_date_summary.csv`: fixed `+/-5` versus optimal result for every cell/date.
- `figures/row_recommended_window_days.png`: quick latitude-band summary.
- `figures/row_date_optimal_window_heatmap.png`: how optimal windows vary over the year.
- `utm_cell_summary.geojson`: spatial layer for GIS inspection.